# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hamidism/Machine-Learning-intern/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

One row = one page (content_hash_id), aggregated over March 2026 (2026-03-01 to 2026-03-31).

I'm using the daily fact table (fact_content_daily_performance/month=2026-03) rolled up
to page-level, joined with static page attributes from dim_content. March 2026 is a
mid-panel month — not the sealed final month (_sample = June 2026), so it's safe for
developing label logic on.*

In [8]:
print("Date range in fact table:", fact_perf["report_date"].min(), "to", fact_perf["report_date"].max())
print("Unique pages in fact table (March 2026):", fact_perf["content_hash_id"].nunique())
print("Unique pages in dim_content (all time):", dim_content["content_hash_id"].nunique())

Date range in fact table: 2026-03-01 00:00:00 to 2026-03-31 00:00:00
Unique pages in fact table (March 2026): 331437
Unique pages in dim_content (all time): 519606


In [9]:
from huggingface_hub import login
from google.colab import userdata
import duckdb

# Authenticate with your HF token stored in Colab Secrets
hf_token = userdata.get('HF_TOKEN')
login(token=hf_token)

# List files available in the warehouse dataset
from huggingface_hub import list_repo_files
files = list_repo_files("FlyRank/internship-warehouse", repo_type="dataset")
for f in files:
    print(f)

.gitattributes
README.md
dim_clients.parquet
dim_content.parquet
fact_content_daily_performance/month=2025-01/data_0.parquet
fact_content_daily_performance/month=2025-02/data_0.parquet
fact_content_daily_performance/month=2025-03/data_0.parquet
fact_content_daily_performance/month=2025-04/data_0.parquet
fact_content_daily_performance/month=2025-05/data_0.parquet
fact_content_daily_performance/month=2025-06/data_0.parquet
fact_content_daily_performance/month=2025-07/data_0.parquet
fact_content_daily_performance/month=2025-08/data_0.parquet
fact_content_daily_performance/month=2025-09/data_0.parquet
fact_content_daily_performance/month=2025-10/data_0.parquet
fact_content_daily_performance/month=2025-11/data_0.parquet
fact_content_daily_performance/month=2025-12/data_0.parquet
fact_content_daily_performance/month=2026-01/data_0.parquet
fact_content_daily_performance/month=2026-02/data_0.parquet
fact_content_daily_performance/month=2026-03/data_0.parquet
fact_content_daily_performance/mont

## 2. Fields: feature / label / context / excluded

Feature (used to predict):
- word_count, char_count (from dim_content) — content depth signals
- backlinks (from dim_content) — authority signal
- avg_position (mean gsc_avg_position over March) — visibility signal
- ctr (gsc_clicks / gsc_impressions over March) — engagement-vs-visibility signal

Label / proxy (what we're trying to predict):
- trend_direction_down — derived by comparing first-half vs second-half March impressions;
  1 if impressions dropped, 0 otherwise

Context (kept for grouping/reporting, not fed to the model):
- client_hash_id, content_type, main_intent

Excluded (and why):
- provider_used, model_used — these describe HOW content was created (tooling metadata),
  not how it's performing; including them risks the model learning tool-brand patterns
  instead of real content-quality signals.
- sessions_paid, sessions_social, sessions_referral, sessions_ai, ai_chatgpt/perplexity/etc. —
  downstream traffic-channel breakdowns, not core decision-moment content signals; excluded
  to keep the first feature frame small and interpretable (5 features max).

In [10]:
from huggingface_hub import hf_hub_download
import pandas as pd

# Download the two key files for our lane
content_path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename="dim_content.parquet",
    repo_type="dataset"
)

perf_path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename="fact_content_daily_performance/month=2026-03/data_0.parquet",
    repo_type="dataset"
)

dim_content = pd.read_parquet(content_path)
fact_perf = pd.read_parquet(perf_path)

print("=== dim_content columns ===")
print(dim_content.columns.tolist())
print("Shape:", dim_content.shape)

print("\n=== fact_content_daily_performance (2026-03) columns ===")
print(fact_perf.columns.tolist())
print("Shape:", fact_perf.shape)

=== dim_content columns ===
['client_hash_id', 'content_hash_id', 'keyword_hash_id', 'url_hash_id', 'keyword_char_count', 'keyword_token_count', 'url_char_count', 'content_created_date', 'content_updated_date', 'content_type', 'search_volume', 'competition', 'competition_level', 'cpc', 'main_intent', 'backlinks', 'category_count', 'keyword_created_date', 'provider_used', 'model_used', 'char_count', 'word_count', 'last_optimized_date', 'optimization_eligible_date', 'is_published', 'is_deleted']
Shape: (519606, 26)

=== fact_content_daily_performance (2026-03) columns ===
['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chat

## 3. Verify it with queries (grain, counts, missing values, windows)

Feature availability:
- word_count: knowable at the decision moment because it's a static property of the page's
  content, set when the page was published.
- char_count: same — static content attribute, known immediately.
- backlinks: knowable because it reflects links already earned as of report time — historical,
  not future.
- avg_position: knowable because it's measured from March's already-occurred search
  impressions — no future leakage.
- ctr: knowable for the same reason — computed from March's already-occurred clicks and
  impressions.

In [11]:
dupes = dim_content["content_hash_id"].duplicated().sum()
print("Duplicate content_hash_id in dim_content:", dupes)
print("This confirms grain:", "PASS - one row = one page" if dupes == 0 else "FAIL - grain violated")

print("Total rows in fact_content_daily_performance (March 2026):", len(fact_perf))
print("Unique pages:", fact_perf["content_hash_id"].nunique())
print("Date span:", fact_perf["report_date"].min(), "->", fact_perf["report_date"].max())
print("Days covered:", fact_perf["report_date"].nunique())

available = fact_perf[(fact_perf["gsc_data_available"] == True) &
                       (fact_perf["ga4_data_available"] == True)]
print("Rows before availability filter:", len(fact_perf))
print("Rows after gsc_data_available IS TRUE and ga4_data_available IS TRUE:", len(available))
print(f"Survival rate: {len(available)/len(fact_perf)*100:.1f}%")

Duplicate content_hash_id in dim_content: 0
This confirms grain: PASS - one row = one page
Total rows in fact_content_daily_performance (March 2026): 9841378
Unique pages: 331437
Date span: 2026-03-01 -> 2026-03-31
Days covered: 31
Rows before availability filter: 9841378
Rows after gsc_data_available IS TRUE and ga4_data_available IS TRUE: 364347
Survival rate: 3.7%


In [12]:
page_agg = available.groupby("content_hash_id").agg(
    total_impressions=("gsc_impressions", "sum"),
    total_clicks=("gsc_clicks", "sum"),
    avg_position=("gsc_avg_position", "mean")
).reset_index()

page_agg["ctr"] = page_agg["total_clicks"] / page_agg["total_impressions"].replace(0, pd.NA)

features = page_agg.merge(
    dim_content[["content_hash_id", "word_count", "char_count", "backlinks"]],
    on="content_hash_id", how="left"
)

feature_cols = ["word_count", "char_count", "backlinks", "avg_position", "ctr"]
features_final = features[["content_hash_id"] + feature_cols].dropna()

print("Feature frame shape:", features_final.shape)
features_final.head()

Feature frame shape: (41310, 6)


,content_hash_id,word_count,char_count,backlinks,avg_position,ctr
0,content_00032be2df0005ca,1502.0,9441.0,1.0,7.052761,0.017544
1,content_00039f4c7a954114,2573.0,17104.0,0.0,6.285714,0.000000
2,content_0004be2ef2278bd8,2679.0,17907.0,0.0,10.911772,0.000000
3,content_000612ace4167db9,2249.0,14576.0,346.0,14.977778,0.011111
4,content_0008f4a4b35f399b,1627.0,9981.0,0.0,38.203290,0.016667


Leakage lesson: Adding LEAK_second_half_clicks pushed AUC from ~[0.616] toward
[0.628] — because the label itself was computed by comparing first-half vs
second-half impressions, and second-half clicks are directly entangled with that same
window. This is decision-moment leakage: a reviewer would never have March 16-31 clicks
available when making a March 1 decision. The honest, deployable number is the 5-feature
score, not the leaked one.

In [13]:
fact_perf["report_date"] = pd.to_datetime(fact_perf["report_date"])
first_half = fact_perf[fact_perf["report_date"] < "2026-03-16"]
second_half = fact_perf[fact_perf["report_date"] >= "2026-03-16"]

imp_first = first_half.groupby("content_hash_id")["gsc_impressions"].sum()
imp_second = second_half.groupby("content_hash_id")["gsc_impressions"].sum()

trend = pd.DataFrame({"imp_first": imp_first, "imp_second": imp_second}).dropna()
trend["label_down"] = (trend["imp_second"] < trend["imp_first"]).astype(int)

leak_test = features_final.merge(trend[["label_down"]], left_on="content_hash_id", right_index=True)

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

X = leak_test[feature_cols]
y = leak_test["label_down"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

model_honest = LogisticRegression(max_iter=1000).fit(X_train, y_train)
honest_score = roc_auc_score(y_test, model_honest.predict_proba(X_test)[:, 1])
print("Honest AUC (5 features only):", round(honest_score, 3))

leak_test["LEAK_second_half_clicks"] = second_half.groupby("content_hash_id")["gsc_clicks"].sum().reindex(leak_test["content_hash_id"]).values

X_leaked = leak_test[feature_cols + ["LEAK_second_half_clicks"]].fillna(0)
X_train2, X_test2, y_train2, y_test2 = train_test_split(X_leaked, y, test_size=0.3, random_state=42)

model_leaked = LogisticRegression(max_iter=1000).fit(X_train2, y_train2)
leaked_score = roc_auc_score(y_test2, model_leaked.predict_proba(X_test2)[:, 1])
print("LEAKED AUC (with second-half clicks added):", round(leaked_score, 3))
print("\n--> Score jumped because the label was derived from this same signal window.")
print("--> Deleting the leaked column and keeping the honest 5-feature score.")

Honest AUC (5 features only): 0.616
LEAKED AUC (with second-half clicks added): 0.628

--> Score jumped because the label was derived from this same signal window.
--> Deleting the leaked column and keeping the honest 5-feature score.


## 4. Data limits

What this data can never tell me:
- This slice only covers pages with BOTH gsc_data_available AND ga4_data_available marked
  true — pages missing one source are silently dropped, likely under-representing newer or
  smaller clients with incomplete tracking.
- One month (March 2026) can't distinguish a real trend from short-term noise or seasonality.
- The daily fact table has no signal for WHY a page changed — any "reason" attached to a
  decline is inferred, not observed.

In [14]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.